# ЛР № 3. Профилирование и качество данных Seoul Bike

Демонстрация участка `raw → staging`. Одна строка описывает **один час работы системы
велопроката** в Сеуле.

Вопрос из [Project Brief](../docs/project_brief.md): как различается почасовой спрос на
велосипеды по сезону, часу суток и типу дня (рабочий, выходной, праздничный)?

Надёжного бизнес-ключа в источнике нет, поэтому проверяется уникальность сочетания
даты и часа: это подтверждение заявленной гранулярности набора, а не доказательство
уникальности реальных событий.

Запустите лабораторный стенд (`docker compose up -d`) и выберите ядро из `.venv` проекта.
Выполняйте ячейки сверху вниз. Для проверки воспроизводимости перезапустите ядро и
выполните все ячейки ещё раз: количества строк должны совпасть.

<a id="input"></a>
## Шаг 1. Входные данные и подключение

`ingested_on` — логическая дата загрузки в слой `raw`, а не период наблюдений.
Аналитический период — весь набор: с 01.12.2017 по 30.11.2018, поэтому отдельный фильтр
по периоду не применяется.

Особенность источника: файл опубликован в кодировке cp1252, а имена температурных полей
содержат знак градуса. Поэтому заголовок файла пропускается, а имена столбцов задаются
явно и сразу в удобном для анализа виде. Соответствие исходных и рабочих имён приведено
в отчёте `docs/data_quality_report.md`.

Следующая ячейка определяет корень проекта при запуске из корня или каталога `notebooks`,
загружает настройки из `.env` без вывода секретов и использует подключение DuckDB из ЛР № 2.

In [1]:
import os
import sys
from html import escape
from pathlib import Path

import boto3
import duckdb
from dotenv import load_dotenv
from IPython.display import HTML, display

PROJECT_ROOT = next(
    parent for parent in (Path.cwd(), *Path.cwd().parents)
    if (parent / "pyproject.toml").is_file()
    and (parent / "src/analysis/read_raw.py").is_file()
)
load_dotenv(PROJECT_ROOT / ".env", override=True)
required = ("S3_ENDPOINT", "S3_ACCESS_KEY", "S3_SECRET_KEY")
missing = [name for name in required if not os.environ.get(name)]
if missing:
    raise ValueError("Не заданы настройки: " + ", ".join(missing))

sys.path.insert(0, str(PROJECT_ROOT / "src"))
from analysis.read_raw import configure_s3  # noqa: E402

# ---- Параметры запуска ---------------------------------------------------
# ingested_on — логическая дата загрузки; должна совпадать с объектом в слое raw.
raw_path = "s3://raw/seoul_bike/ingested_on=2026-11-12/SeoulBikeData.csv"
source_date = raw_path.split("/ingested_on=", 1)[1].split("/", 1)[0]
output_prefix = f"s3://staging/seoul_bike/source_ingested_on={source_date}"

# Рабочие имена столбцов. Заголовок исходного файла пропускается: он в cp1252.
COLUMN_NAMES = (
    "observed_on", "rented_bike_count", "hour", "temperature_c", "humidity_pct",
    "wind_speed_ms", "visibility_10m", "dew_point_c", "solar_radiation_mj_m2",
    "rainfall_mm", "snowfall_cm", "season", "holiday", "functioning_day",
)
CSV_HEADER_SKIP = 1


def literal(value):
    return "'" + str(value).replace("'", "''") + "'"


def show_table(headers, rows):
    header = "".join(f"<th>{escape(str(item))}</th>" for item in headers)
    body = "".join(
        "<tr>" + "".join(f"<td>{escape(str(item))}</td>" for item in row) + "</tr>"
        for row in rows
    )
    display(HTML(f"<table><thead><tr>{header}</tr></thead><tbody>{body}</tbody></table>"))


def raw_relation():
    names = ", ".join(literal(name) for name in COLUMN_NAMES)
    return (
        f"read_csv_auto({literal(raw_path)}, skip={CSV_HEADER_SKIP}, header=false, "
        f"column_names=[{names}], auto_detect=true)"
    )


connection = duckdb.connect()
configure_s3(connection)
connection.execute(f"CREATE OR REPLACE TEMP VIEW raw_data AS SELECT * FROM {raw_relation()}")

show_table(
    ["Вход", "Логическая дата", "Выходной префикс"],
    [(raw_path, source_date, output_prefix)],
)
show_table(
    ["Дата", "Час", "Аренд", "Темп.", "Сезон", "Праздник", "Сервис"],
    connection.execute("""
        SELECT observed_on, hour, rented_bike_count, temperature_c,
               season, holiday, functioning_day
        FROM raw_data ORDER BY observed_on, hour LIMIT 3
    """).fetchall(),
)

Вход,Логическая дата,Выходной префикс
s3://raw/seoul_bike/ingested_on=2026-11-12/SeoulBikeData.csv,2026-11-12,s3://staging/seoul_bike/source_ingested_on=2026-11-12


Дата,Час,Аренд,Темп.,Сезон,Праздник,Сервис
2017-12-01,0,254,-5.2,Winter,No Holiday,True
2017-12-01,1,204,-5.5,Winter,No Holiday,True
2017-12-01,2,173,-6.0,Winter,No Holiday,True


<a id="profile"></a>
## Шаг 2. Профиль исходных данных

Выбраны поля, необходимые для метрик из Project Brief: дата и час для временных срезов,
число аренд как основная метрика, сезон и признак праздника для срезов, признак работы
сервиса для отделения остановки сервиса от нулевого спроса, а также температура и осадки
для уточняющего вопроса о погоде.

Профиль рассчитывается по исходным данным до любых преобразований и исключения записей.

In [2]:
raw_count = connection.execute("SELECT count(*) FROM raw_data").fetchone()[0]
schema = {row[0]: row[1] for row in connection.execute("DESCRIBE raw_data").fetchall()}
show_table(["Строк", "Полей"], [(raw_count, len(schema))])

selected_fields = (
    "observed_on", "hour", "rented_bike_count", "season",
    "holiday", "functioning_day", "temperature_c", "rainfall_mm",
)
profile_rows = []
for field in selected_fields:
    missing_count = connection.execute(
        f"SELECT count(*) FROM raw_data WHERE {field} IS NULL"
    ).fetchone()[0]
    examples = connection.execute(
        f"SELECT DISTINCT {field} FROM raw_data WHERE {field} IS NOT NULL ORDER BY 1 LIMIT 3"
    ).fetchall()
    share = f"{missing_count / raw_count:.3%}" if raw_count else "N/A"
    profile_rows.append((
        field, schema[field], ", ".join(str(row[0]) for row in examples),
        raw_count, missing_count, share,
    ))
show_table(["Поле", "Тип", "Примеры", "Проверено", "Пропусков", "Доля"], profile_rows)

Строк,Полей
8760,15


Поле,Тип,Примеры,Проверено,Пропусков,Доля
observed_on,DATE,"2017-12-01, 2017-12-02, 2017-12-03",8760,0,0.000%
hour,BIGINT,"0, 1, 2",8760,0,0.000%
rented_bike_count,BIGINT,"0, 2, 3",8760,0,0.000%
season,VARCHAR,"Autumn, Spring, Summer",8760,0,0.000%
holiday,VARCHAR,"Holiday, No Holiday",8760,0,0.000%
functioning_day,BOOLEAN,"False, True",8760,0,0.000%
temperature_c,DOUBLE,"-17.8, -17.5, -17.4",8760,0,0.000%
rainfall_mm,DOUBLE,"0.0, 0.1, 0.2",8760,0,0.000%


In [3]:
period = connection.execute(
    "SELECT min(observed_on), max(observed_on) FROM raw_data"
).fetchone()
distinct_days = connection.execute(
    "SELECT count(DISTINCT observed_on) FROM raw_data"
).fetchone()[0]
day_hour_pairs = connection.execute(
    "SELECT count(*) FROM (SELECT DISTINCT observed_on, hour FROM raw_data)"
).fetchone()[0]
exact_duplicates = connection.execute("""
    SELECT count(*) - (SELECT count(*) FROM (SELECT DISTINCT * FROM raw_data))
    FROM raw_data
""").fetchone()[0]
show_table(
    ["Мин. дата", "Макс. дата", "Дней", "Пар дата+час", "Полных повторов"],
    [(*period, distinct_days, day_hour_pairs, exact_duplicates)],
)

days_without_24_hours = connection.execute("""
    SELECT count(*) FROM (
        SELECT observed_on FROM raw_data GROUP BY 1 HAVING count(*) <> 24
    )
""").fetchone()[0]
show_table(["Дней, где число часов отлично от 24"], [(days_without_24_hours,)])

Мин. дата,Макс. дата,Дней,Пар дата+час,Полных повторов
2017-12-01,2018-11-30,365,8760,0


"Дней, где число часов отлично от 24"
0


In [4]:
hours = connection.execute("""
    SELECT hour, count(*) AS rows, round(avg(rented_bike_count), 1) AS avg_rented
    FROM raw_data GROUP BY 1 ORDER BY 1
""").fetchall()
show_table(["Час", "Записей", "Средний спрос"], hours)

service = connection.execute("""
    SELECT functioning_day, count(*) AS rows,
           min(rented_bike_count) AS min_rented,
           max(rented_bike_count) AS max_rented,
           sum(rented_bike_count) AS sum_rented
    FROM raw_data GROUP BY 1 ORDER BY 1
""").fetchall()
show_table(["Сервис работал", "Записей", "Мин. спрос", "Макс. спрос", "Сумма спроса"], service)

zero_demand_by_service = connection.execute("""
    SELECT functioning_day, count(*) AS rows
    FROM raw_data WHERE rented_bike_count = 0 GROUP BY 1 ORDER BY 1
""").fetchall()
show_table(["Сервис работал", "Записей с нулевым спросом"], zero_demand_by_service)

closed_summary = connection.execute("""
    SELECT count(*) AS closed_hours,
           count(DISTINCT observed_on) AS closed_days,
           min(observed_on) AS first_day,
           max(observed_on) AS last_day
    FROM raw_data WHERE NOT functioning_day
""").fetchone()
show_table(
    ["Часов с остановкой сервиса", "Дней с остановкой", "Первый день", "Последний день"],
    [closed_summary],
)

Час,Записей,Средний спрос
0,365,541.5
1,365,426.2
2,365,301.6
3,365,203.3
4,365,132.6
5,365,139.1
6,365,287.6
7,365,606.0
8,365,1015.7
9,365,646.0


Сервис работал,Записей,Мин. спрос,Макс. спрос,Сумма спроса
False,295,0,0,0
True,8465,2,3556,6172314


Сервис работал,Записей с нулевым спросом
False,295


Часов с остановкой сервиса,Дней с остановкой,Первый день,Последний день
295,13,2018-04-11,2018-11-09


### Наблюдения для отчёта

- **Нулевой спрос объясняется остановкой сервиса.** Все записи с `rented_bike_count = 0`
  имеют `functioning_day = false`, и наоборот: при работающем сервисе нулевого спроса не
  встречается ни разу. Значит, ноль описывает закрытый сервис, а не отсутствие желания
  арендовать велосипед. Если такие часы не отделить, средний спрос по срезам окажется
  заниженным: это самое важное наблюдение для дальнейших работ.
- **Пропусков и повторов нет.** Все поля заполнены, полных повторов нет, сочетание
  дата + час уникально, каждый из 365 дней содержит ровно 24 часа.
- **Даты разобраны верно.** Минимум 01.12.2017 и максимум 30.11.2018 совпадают с описанием
  набора, поэтому формат `ДД/ММ/ГГГГ` распознан правильно, а не перепутан с `ММ/ДД/ГГГГ`.
  При обратном чтении период сдвинулся бы, а число дней не совпало бы с 365.
- **Справочники чистые.** Сезон принимает ровно четыре ожидаемых значения и согласован с
  месяцем: зима покрывает декабрь, январь и февраль, весна — март, апрель и май и так далее.
- Профиль выявил два подозрительных значения в погодных полях: влажность 0 % и снег при
  положительной температуре. Они проверяются правилами ниже.

<a id="rules"></a>
## Шаг 3. Правила качества

Для каждого правила ниже указаны идентификатор, поля, проверяемые строки, условие,
критичность, действие и связь с задачей. Порог задан переменной в ячейке и участвует
в сравнении. Единица порога — число нарушений, допустимый максимум — ноль.

Порог оценивает набор целиком. Действие по каждой нарушившей строке применяется
независимо от того, превышен ли порог. Доля считается внутри проверяемой совокупности;
при пустой совокупности доля и сравнение — `N/A`.

Семь правил совместно покрывают уникальность, полноту, допустимость значений и
согласованность связанных полей.

In [5]:
rule_results = {}


def show_rule(rule_id, checked, violations, maximum, action):
    share = f"{violations / checked:.3%}" if checked else "N/A"
    comparison = ("Превышен" if violations > maximum else "Не превышен") if checked else "N/A"
    row = (rule_id, checked, violations, share, f"count <= {maximum}", comparison, action)
    rule_results[rule_id] = row
    show_table(["Правило", "Проверено", "Нарушений", "Доля", "Порог", "Сравнение", "Действие"],
               [row])

### Правило 1. `unique_date_hour_key`

- Поля: `observed_on`, `hour`.
- Проверяемые строки: весь набор.
- Условие: сочетание даты и часа не повторяется.
- Порог: число нарушений; переменная `max_duplicate_keys`.
- Критичность: `critical`.
- Действие: `quarantine`.
- Связь с задачей: заявленная гранулярность — один час одних суток. Повтор пары завысил бы
  число часов и исказил средний спрос по любому срезу вопроса.

Ключ выведен из гранулярности набора. Это подтверждение уникальности часа наблюдения,
а не доказательство уникальности реальных событий: отдельная поездка в данных не видна.

In [6]:
max_duplicate_keys = 0
checked, violations = connection.execute("""
    SELECT count(*), count(*) FILTER (WHERE duplicates > 1) FROM (
        SELECT observed_on, hour, count(*) AS duplicates
        FROM raw_data GROUP BY 1, 2
    )
""").fetchone()
show_rule("unique_date_hour_key", checked, violations, max_duplicate_keys, "quarantine")

Правило,Проверено,Нарушений,Доля,Порог,Сравнение,Действие
unique_date_hour_key,8760,0,0.000%,count <= 0,Не превышен,quarantine


### Правило 2. `required_fields_present`

- Поля: все 14 полей набора.
- Проверяемые строки: весь набор.
- Условие: ни одно из полей не пусто.
- Порог: число нарушений; переменная `max_missing_values`.
- Критичность: `critical`.
- Действие: `quarantine`.
- Связь с задачей: без даты и часа нельзя построить временные срезы, без числа аренд нет
  основной метрики, без признака работы сервиса нельзя отделить остановку сервиса.

Пропусков в этом наборе нет, но правило остаётся: оно фиксирует полноту входных данных
и защищает от молчаливого изменения источника.

In [7]:
max_missing_values = 0
checked, violations = connection.execute("""
    SELECT count(*), count(*) FILTER (
        WHERE observed_on IS NULL OR rented_bike_count IS NULL OR hour IS NULL
           OR temperature_c IS NULL OR humidity_pct IS NULL OR wind_speed_ms IS NULL
           OR visibility_10m IS NULL OR dew_point_c IS NULL
           OR solar_radiation_mj_m2 IS NULL OR rainfall_mm IS NULL
           OR snowfall_cm IS NULL OR season IS NULL OR holiday IS NULL
           OR functioning_day IS NULL
    ) FROM raw_data
""").fetchone()
show_rule("required_fields_present", checked, violations, max_missing_values, "quarantine")

Правило,Проверено,Нарушений,Доля,Порог,Сравнение,Действие
required_fields_present,8760,0,0.000%,count <= 0,Не превышен,quarantine


### Правило 3. `demand_non_negative`

- Поле: `rented_bike_count`.
- Проверяемые строки: весь набор.
- Условие: число аренд неотрицательно.
- Порог: число нарушений; переменная `max_negative_demand`.
- Критичность: `critical`.
- Действие: `quarantine`.
- Связь с задачей: отрицательное число аренд невозможно и исказило бы сумму и среднее
  по всем срезам вопроса — по сезону, часу и типу дня.

In [8]:
max_negative_demand = 0
checked, violations = connection.execute("""
    SELECT count(*), count(*) FILTER (WHERE rented_bike_count < 0)
    FROM raw_data
""").fetchone()
show_rule("demand_non_negative", checked, violations, max_negative_demand, "quarantine")

Правило,Проверено,Нарушений,Доля,Порог,Сравнение,Действие
demand_non_negative,8760,0,0.000%,count <= 0,Не превышен,quarantine


### Правило 4. `hour_within_day`

- Поле: `hour`.
- Проверяемые строки: весь набор.
- Условие: час лежит в диапазоне от 0 до 23.
- Порог: число нарушений; переменная `max_invalid_hour`.
- Критичность: `critical`.
- Действие: `quarantine`.
- Связь с задачей: часы суток — основной срез вопроса. Значение вне диапазона создало бы
  несуществующую группу и сломало сравнение по часам.

In [9]:
max_invalid_hour = 0
checked, violations = connection.execute("""
    SELECT count(*), count(*) FILTER (WHERE hour NOT BETWEEN 0 AND 23)
    FROM raw_data
""").fetchone()
show_rule("hour_within_day", checked, violations, max_invalid_hour, "quarantine")

Правило,Проверено,Нарушений,Доля,Порог,Сравнение,Действие
hour_within_day,8760,0,0.000%,count <= 0,Не превышен,quarantine


### Правило 5. `closed_service_has_no_demand`

- Поля: связь `functioning_day` и `rented_bike_count`.
- Проверяемые строки: часы, когда сервис не работал.
- Условие: если сервис не работал, число аренд равно нулю.
- Порог: число нарушений; переменная `max_closed_with_demand`.
- Критичность: `critical`.
- Действие: `quarantine`.
- Связь с задачей: проверка согласованности двух полей. Если закрытый сервис показывает
  спрос, поля противоречат друг другу и метрика спроса недостоверна.

Правило не удаляет часы остановки сервиса: они остаются в `staging` с признаком
`service_closed`, потому что описывают реальное состояние системы, а не ошибку данных.

In [10]:
max_closed_with_demand = 0
checked, violations = connection.execute("""
    SELECT count(*) FILTER (WHERE NOT functioning_day),
           count(*) FILTER (WHERE NOT functioning_day AND rented_bike_count <> 0)
    FROM raw_data
""").fetchone()
show_rule("closed_service_has_no_demand", checked, violations, max_closed_with_demand, "quarantine")

Правило,Проверено,Нарушений,Доля,Порог,Сравнение,Действие
closed_service_has_no_demand,295,0,0.000%,count <= 0,Не превышен,quarantine


### Правило 6. `humidity_zero_is_suspicious`

- Поле: `humidity_pct`.
- Проверяемые строки: весь набор.
- Условие: относительная влажность больше нуля.
- Порог: число нарушений; переменная `max_zero_humidity`.
- Критичность: `warning`.
- Действие: `flag`.
- Связь с задачей: влажность входит в уточняющий вопрос о связи погоды и спроса. Нулевая
  влажность на открытом воздухе неправдоподобна и похожа на сбой датчика, поэтому такие
  часы нужно видеть отдельно и исключать из расчёта связи по влажности.

Действие `flag`: запись остаётся в `staging` с явным признаком. Удалять её нельзя — число
аренд в этом часе достоверно, сомнителен только погодный показатель.

In [11]:
max_zero_humidity = 0
checked, violations = connection.execute("""
    SELECT count(*), count(*) FILTER (WHERE humidity_pct = 0)
    FROM raw_data
""").fetchone()
show_rule("humidity_zero_is_suspicious", checked, violations, max_zero_humidity, "flag")

Правило,Проверено,Нарушений,Доля,Порог,Сравнение,Действие
humidity_zero_is_suspicious,8760,17,0.194%,count <= 0,Превышен,flag


### Правило 7. `snowfall_requires_cold`

- Поля: связь `snowfall_cm` и `temperature_c`.
- Проверяемые строки: весь набор.
- Условие: если зафиксирован снег, температура ниже 5 °C.
- Порог: число нарушений; переменная `max_warm_snowfall`.
- Критичность: `warning`.
- Действие: `flag`.
- Связь с задачей: осадки и температура участвуют в уточняющем вопросе. Снег при
  температуре выше 5 °C противоречит физике и указывает на возможную ошибку единиц
  измерения или сдвиг времени наблюдения.

Порог выбран равным 5 °C, а не 0 °C: снег при слабоположительной температуре у поверхности
возможен, поэтому строгий порог дал бы ложные срабатывания. Записи сохраняются с признаком.

In [12]:
max_warm_snowfall = 0
checked, violations = connection.execute("""
    SELECT count(*), count(*) FILTER (WHERE snowfall_cm > 0 AND temperature_c >= 5)
    FROM raw_data
""").fetchone()
show_rule("snowfall_requires_cold", checked, violations, max_warm_snowfall, "flag")

Правило,Проверено,Нарушений,Доля,Порог,Сравнение,Действие
snowfall_requires_cold,8760,14,0.160%,count <= 0,Превышен,flag


### Сводка по всем правилам

Таблица собрана из результатов проверок выше. Критических нарушений в наборе нет,
поэтому карантин окажется пустым. Два правила-предупреждения нашли записи: они остаются
в `staging` с явными признаками.

In [13]:
show_table(["Правило", "Проверено", "Нарушений", "Доля", "Порог", "Сравнение", "Действие"],
           rule_results.values())

Правило,Проверено,Нарушений,Доля,Порог,Сравнение,Действие
unique_date_hour_key,8760,0,0.000%,count <= 0,Не превышен,quarantine
required_fields_present,8760,0,0.000%,count <= 0,Не превышен,quarantine
demand_non_negative,8760,0,0.000%,count <= 0,Не превышен,quarantine
hour_within_day,8760,0,0.000%,count <= 0,Не превышен,quarantine
closed_service_has_no_demand,295,0,0.000%,count <= 0,Не превышен,quarantine
humidity_zero_is_suspicious,8760,17,0.194%,count <= 0,Превышен,flag
snowfall_requires_cold,8760,14,0.160%,count <= 0,Превышен,flag


<a id="prepare"></a>
## Шаг 4. Подготовка `staging` и карантина

По результатам проверок критические правила не нарушены ни разу, поэтому карантин
останется пустым — файл всё равно создаётся, с определённой схемой.

Что происходит с данными:

- Имена и типы полей приведены к рабочим при чтении (`observed_on`, `rented_bike_count`,
  `hour`, `temperature_c`, ...). Переименование задаётся явно в ячейке параметров, потому
  что исходные имена содержат пробелы, скобки и знак градуса и неудобны в SQL.
- Записи с `flag`-нарушениями остаются в `staging` и получают явный признак нарушения:
  `quality_humidity_zero_is_suspicious_violated` и `quality_snowfall_requires_cold_violated`.
- Добавляется описательный признак `service_closed` — час остановки сервиса. Это не признак
  нарушения, а характеристика записи, важная для дальнейшего анализа: такие часы нельзя
  смешивать с обычными при расчёте среднего спроса.
- Бизнес-агрегаций и итоговых метрик в `staging` нет.

Если запись нарушила несколько критических правил, она попадёт в карантин один раз, а все
причины сохранятся. Карантин имеет приоритет над признаками `flag`.

In [14]:
connection.execute("""
    CREATE OR REPLACE TEMP VIEW checked_data AS
    SELECT *,
        observed_on IS NULL OR hour IS NULL OR rented_bike_count IS NULL
            OR temperature_c IS NULL OR humidity_pct IS NULL OR wind_speed_ms IS NULL
            OR visibility_10m IS NULL OR dew_point_c IS NULL
            OR solar_radiation_mj_m2 IS NULL OR rainfall_mm IS NULL
            OR snowfall_cm IS NULL OR season IS NULL OR holiday IS NULL
            OR functioning_day IS NULL AS missing_required,
        rented_bike_count < 0 AS negative_demand,
        hour NOT BETWEEN 0 AND 23 AS invalid_hour,
        COALESCE(NOT functioning_day AND rented_bike_count <> 0, false) AS closed_with_demand,
        COALESCE(humidity_pct = 0, false) AS zero_humidity,
        COALESCE(snowfall_cm > 0 AND temperature_c >= 5, false) AS warm_snowfall
    FROM raw_data
""")

connection.execute("""
    CREATE OR REPLACE TEMP VIEW duplicate_key_data AS
    SELECT observed_on, hour FROM raw_data GROUP BY 1, 2 HAVING count(*) > 1
""")

connection.execute("""
    CREATE OR REPLACE TEMP VIEW staging_data AS
    SELECT
        c.* EXCLUDE (missing_required, negative_demand, invalid_hour, closed_with_demand,
                     zero_humidity, warm_snowfall),
        NOT c.functioning_day AS service_closed,
        c.zero_humidity AS quality_humidity_zero_is_suspicious_violated,
        c.warm_snowfall AS quality_snowfall_requires_cold_violated
    FROM checked_data AS c
    WHERE NOT (c.missing_required OR c.negative_demand OR c.invalid_hour
               OR c.closed_with_demand)
      AND NOT EXISTS (
          SELECT 1 FROM duplicate_key_data AS d
          WHERE d.observed_on = c.observed_on AND d.hour = c.hour
      )
""")

connection.execute("""
    CREATE OR REPLACE TEMP VIEW quarantine_data AS
    SELECT
        c.* EXCLUDE (missing_required, negative_demand, invalid_hour, closed_with_demand,
                     zero_humidity, warm_snowfall),
        concat(
            CASE WHEN c.missing_required THEN 'required_fields_present;' ELSE '' END,
            CASE WHEN c.negative_demand THEN 'demand_non_negative;' ELSE '' END,
            CASE WHEN c.invalid_hour THEN 'hour_within_day;' ELSE '' END,
            CASE WHEN c.closed_with_demand THEN 'closed_service_has_no_demand;' ELSE '' END,
            CASE WHEN EXISTS (
                SELECT 1 FROM duplicate_key_data AS d
                WHERE d.observed_on = c.observed_on AND d.hour = c.hour
            ) THEN 'unique_date_hour_key;' ELSE '' END
        ) AS quality_rule_ids,
        concat(
            CASE WHEN c.missing_required THEN 'Не заполнено обязательное поле; ' ELSE '' END,
            CASE WHEN c.negative_demand THEN 'Отрицательное число аренд; ' ELSE '' END,
            CASE WHEN c.invalid_hour THEN 'Час вне диапазона 0-23; ' ELSE '' END,
            CASE WHEN c.closed_with_demand THEN 'Сервис не работал, но есть спрос; ' ELSE '' END,
            CASE WHEN EXISTS (
                SELECT 1 FROM duplicate_key_data AS d
                WHERE d.observed_on = c.observed_on AND d.hour = c.hour
            ) THEN 'Повтор сочетания даты и часа; ' ELSE '' END
        ) AS quarantine_reason
    FROM checked_data AS c
    WHERE c.missing_required OR c.negative_demand OR c.invalid_hour
       OR c.closed_with_demand
       OR EXISTS (
           SELECT 1 FROM duplicate_key_data AS d
           WHERE d.observed_on = c.observed_on AND d.hour = c.hour
       )
""")

show_table(["Набор", "Строк"], [
    (name, connection.execute(f"SELECT count(*) FROM {name}_data").fetchone()[0])
    for name in ("raw", "staging", "quarantine")
])

Набор,Строк
raw,8760
staging,8760
quarantine,0


### Примеры записей

Показаны строки, получившие признаки нарушений, и одна запись об остановке сервиса.
В карантине при пустом результате таблица не выводится.

In [15]:
show_table(
    ["Дата", "Час", "Аренд", "Влажн.", "Флаг влажности"],
    connection.execute("""
        SELECT observed_on, hour, rented_bike_count, humidity_pct,
               quality_humidity_zero_is_suspicious_violated
        FROM staging_data WHERE quality_humidity_zero_is_suspicious_violated
        ORDER BY observed_on, hour LIMIT 3
    """).fetchall(),
)
show_table(
    ["Дата", "Час", "Аренд", "Снег", "Темп.", "Флаг снега"],
    connection.execute("""
        SELECT observed_on, hour, rented_bike_count, snowfall_cm, temperature_c,
               quality_snowfall_requires_cold_violated
        FROM staging_data WHERE quality_snowfall_requires_cold_violated
        ORDER BY observed_on, hour LIMIT 3
    """).fetchall(),
)
show_table(
    ["Дата", "Час", "Аренд", "Сервис закрыт"],
    connection.execute("""
        SELECT observed_on, hour, rented_bike_count, service_closed
        FROM staging_data WHERE service_closed
        ORDER BY observed_on, hour LIMIT 3
    """).fetchall(),
)

quarantine_preview = connection.execute("""
    SELECT observed_on, hour, rented_bike_count, quality_rule_ids, quarantine_reason
    FROM quarantine_data ORDER BY observed_on, hour LIMIT 3
""").fetchall()
if quarantine_preview:
    show_table(["Дата", "Час", "Аренд", "Правила", "Причина"], quarantine_preview)
else:
    show_table(["Карантин"], [("пусто: критических нарушений нет",)])

Дата,Час,Аренд,Влажн.,Флаг влажности
2018-05-19,7,436,0,True
2018-05-21,2,262,0,True
2018-05-21,3,165,0,True


Дата,Час,Аренд,Снег,Темп.,Флаг снега
2017-12-10,15,180,1.0,6.7,True
2017-12-10,16,214,0.5,5.4,True
2018-02-23,12,283,1.7,5.3,True


Дата,Час,Аренд,Сервис закрыт
2018-04-11,0,0,True
2018-04-11,1,0,True
2018-04-11,2,0,True


Карантин
пусто: критических нарушений нет


### Запись двух результатов в S3

Следующая ячейка заменяет только `staging.parquet` и `quarantine.parquet` по префиксу,
собранному из логической даты исходного raw-объекта. Префикс проверяется: он обязан
соответствовать дате в пути входного файла, а не текущей дате запуска.

In [16]:
expected_prefix = f"s3://staging/seoul_bike/source_ingested_on={source_date}"
if output_prefix != expected_prefix:
    raise ValueError("Выходной префикс не соответствует исходному снимку.")

endpoint = os.environ["S3_ENDPOINT"]
client = boto3.client(
    "s3",
    endpoint_url=endpoint if endpoint.startswith("http") else f"http://{endpoint}",
    aws_access_key_id=os.environ["S3_ACCESS_KEY"],
    aws_secret_access_key=os.environ["S3_SECRET_KEY"],
    region_name="us-east-1",
)
prefix = output_prefix.removeprefix("s3://staging/")
for name in ("staging", "quarantine"):
    key = f"{prefix}/{name}.parquet"
    client.delete_object(Bucket="staging", Key=key)
    connection.execute(
        f"COPY (SELECT * FROM {name}_data) TO {literal('s3://staging/' + key)} (FORMAT PARQUET)"
    )
show_table(["Результат", "S3-путь"], [
    (name, f"{output_prefix}/{name}.parquet") for name in ("staging", "quarantine")
])

Результат,S3-путь
staging,s3://staging/seoul_bike/source_ingested_on=2026-11-12/staging.parquet
quarantine,s3://staging/seoul_bike/source_ingested_on=2026-11-12/quarantine.parquet


<a id="verify"></a>
## Шаг 5. Проверка сохранённых файлов

Читаем оба файла непосредственно из S3 и проверяем: баланс строк, отсутствие критических
нарушений в `staging`, правильность признаков и наличие причин в карантине.

`raw = staging + карантин` — записи не теряются и не добавляются при обработке.

In [17]:
for name in ("staging", "quarantine"):
    path = f"{output_prefix}/{name}.parquet"
    connection.execute(
        f"CREATE OR REPLACE TEMP VIEW saved_{name} AS SELECT * FROM read_parquet({literal(path)})"
    )

staging_count = connection.execute("SELECT count(*) FROM saved_staging").fetchone()[0]
quarantine_count = connection.execute("SELECT count(*) FROM saved_quarantine").fetchone()[0]

critical_count = connection.execute("""
    SELECT count(*) FROM saved_staging
    WHERE observed_on IS NULL OR hour IS NULL OR rented_bike_count IS NULL
       OR temperature_c IS NULL OR humidity_pct IS NULL OR wind_speed_ms IS NULL
       OR visibility_10m IS NULL OR dew_point_c IS NULL
       OR solar_radiation_mj_m2 IS NULL OR rainfall_mm IS NULL
       OR snowfall_cm IS NULL OR season IS NULL OR holiday IS NULL
       OR functioning_day IS NULL
       OR rented_bike_count < 0
       OR hour NOT BETWEEN 0 AND 23
       OR (NOT functioning_day AND rented_bike_count <> 0)
""").fetchone()[0]

bad_humidity_flags = connection.execute("""
    SELECT count(*) FROM saved_staging
    WHERE quality_humidity_zero_is_suspicious_violated
          IS DISTINCT FROM COALESCE(humidity_pct = 0, false)
""").fetchone()[0]
bad_snow_flags = connection.execute("""
    SELECT count(*) FROM saved_staging
    WHERE quality_snowfall_requires_cold_violated
          IS DISTINCT FROM COALESCE(snowfall_cm > 0 AND temperature_c >= 5, false)
""").fetchone()[0]
bad_service_flags = connection.execute("""
    SELECT count(*) FROM saved_staging
    WHERE service_closed IS DISTINCT FROM NOT functioning_day
""").fetchone()[0]

duplicate_keys = connection.execute("""
    SELECT count(*) FROM (
        SELECT observed_on, hour FROM saved_staging GROUP BY 1, 2 HAVING count(*) > 1
    )
""").fetchone()[0]

bad_reasons = connection.execute("""
    SELECT count(*) FROM saved_quarantine
    WHERE quality_rule_ids IS NULL OR quality_rule_ids = ''
       OR quarantine_reason IS NULL OR quarantine_reason = ''
""").fetchone()[0]

assert raw_count == staging_count + quarantine_count, "Не сошёлся баланс строк."
assert critical_count == 0, "В staging остались критические нарушения."
assert bad_humidity_flags == 0, "Неверный признак нулевой влажности."
assert bad_snow_flags == 0, "Неверный признак снега при тёплой температуре."
assert bad_service_flags == 0, "Неверный признак остановки сервиса."
assert duplicate_keys == 0, "В staging повторяется сочетание даты и часа."
assert bad_reasons == 0, "В карантине есть строки без причины."

flagged_humidity = connection.execute("""
    SELECT count(*) FROM saved_staging WHERE quality_humidity_zero_is_suspicious_violated
""").fetchone()[0]
flagged_snow = connection.execute("""
    SELECT count(*) FROM saved_staging WHERE quality_snowfall_requires_cold_violated
""").fetchone()[0]
service_closed_count = connection.execute(
    "SELECT count(*) FROM saved_staging WHERE service_closed"
).fetchone()[0]

run_summary = (raw_count, staging_count, quarantine_count, flagged_humidity,
               flagged_snow, service_closed_count, critical_count)
show_table(
    ["Raw", "Staging", "Карантин", "Флаг влажности", "Флаг снега",
     "Часов закрытого сервиса", "Критические нарушения"],
    [run_summary],
)
show_table(["Проверка", "Нарушений"], [
    ("Неверные признаки влажности", bad_humidity_flags),
    ("Неверные признаки снега", bad_snow_flags),
    ("Неверные признаки сервиса", bad_service_flags),
    ("Повторы ключа в staging", duplicate_keys),
    ("Пустые причины карантина", bad_reasons),
])

Raw,Staging,Карантин,Флаг влажности,Флаг снега,Часов закрытого сервиса,Критические нарушения
8760,8760,0,17,14,295,0


Проверка,Нарушений
Неверные признаки влажности,0
Неверные признаки снега,0
Неверные признаки сервиса,0
Повторы ключа в staging,0
Пустые причины карантина,0


### Повторное выполнение с чистого ядра

Запишите количества первого выполнения в отчёт. Перезапустите ядро и выполните **все ячейки
сверху вниз** с теми же входным путём, порогами и выходным префиксом. Количества строк
в `staging` и карантине должны совпасть, а дубликаты не должны появиться.

Контрольные количества для этого снимка: raw — 8760, staging — 8760, карантин — 0,
флаг нулевой влажности — 17, флаг снега при тёплой температуре — 14,
часов закрытого сервиса — 295, критические нарушения в staging — 0.